# CryptoHypothesisLab — 04 / V2: jednokratna priprema C1 OHLCV + sekundnih C2 agregata

**Ovo je samo vlasnički preparer.** Ugrađeni Google Drive mount dozvoljen je samo ovdje. Izvorni `CryptoResearchLab` ostaje bez izmjena. Code-only javni repo ne sadrži podatke. **Ne pokretati analizu u ovoj sesiji.**

- Obavezno novo odobrenje: `AUTHORIZE_MICRO_DEVELOPMENT_YYYY-MM`, nakon frozen preflighta.
- C1: svi mjesečni 1m OHLCV/taker kolone; C2: streaming jednim prolazom u rijetke 5s/15s/60s intervale. Nema order-book C3 i nema sirovih aggTrades u izlaznom paketu.
- Rezultat ZIP je privatan, može biti desetine MB; čuvaj ga samo lokalno i ne objavljuj GitHubu. Podaci nisu ishod transakcija ni dokaz izvršne cijene.
- Colab Drive mount i FUSE event nisu nezavisno trajni, atomarni dokaz istraživačkog prava. Ovo je vlasnički kontrolisani manuelni pilot.
- Ako prijenos ne uspije, događaj ostaje evidentiran; bez automatskog nastavka ili brisanja prethodnih tragova.


In [ ]:
# 1. Javni kod i pinned runtime, PRIJE Drive mounta.
import subprocess,sys,secrets,os
from pathlib import Path
repo=Path('/content/hypolab-micro-source')
if repo.exists(): raise RuntimeError('Koristi novu Colab sesiju, ne stari checkout.')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
subprocess.run([sys.executable,'-m','pip','-q','install','pyarrow==26.0.0'],check=True)
sys.path.insert(0,str(repo/'src'))
print('V2 kod commit:',commit)


In [ ]:
# 2. Vlasnički Drive mount + samo frozen metadata preflight.
from google.colab import drive
from hypolab.frozen import FrozenIndex
from hypolab.colab_micro_prep import prepare_one_micro_month

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive')
metadata_parent=base/'CryptoResearchLab'/'BTC-DATA-UNIVERSE-V1-REVIEW-20261108'/'BTC_DATA_UNIVERSE_V1_FROZEN'
source_generations=base/'CryptoResearchLab'/'data'/'canonical'/'generations'
private_events=base/'CryptoHypothesisLab'/'research-events'
universe_sha=input('Frozen Universe SHA-256: ').strip()
calendar_sha=input('Frozen Calendar SHA-256: ').strip()
metadata_root=metadata_parent/universe_sha
frozen=FrozenIndex.load(metadata_root,universe_sha,calendar_sha)
print('Frozen metadata PASS. C1/C2 partitions:',sum(p[0]=='klines' for p in frozen.partitions),sum(p[0]=='aggTrades' for p in frozen.partitions))
print('Nijedan market-data red jos nije otvoren.')


In [ ]:
# 3. NOVO vlasničko odobrenje samo jednog Development mjeseca.
month=input('Izaberi 2020-2024 mjesec (npr. 2024-07): ').strip()
if month not in {f'{y}-{m:02d}' for y in range(2020,2025) for m in range(1,13)}:
    raise RuntimeError('Validation/OOS/month not allowed')
confirmation=input(f'Upisi tacno AUTHORIZE_MICRO_DEVELOPMENT_{month}: ').strip()
attempt_id=secrets.token_hex(8)
cache=Path('/content/hypolab-micro-cache')
scratch=Path('/content/hypolab-micro-scratch')
packets=Path('/content/hypolab-micro-packets')
for folder in (cache,scratch,packets):folder.mkdir(mode=0o700,exist_ok=False)
result=prepare_one_micro_month(metadata_root=metadata_root,
    generations_root=source_generations,events_root=private_events,
    private_cache_parent=cache,private_scratch=scratch,private_packets=packets,
    month=month,attempt_id=attempt_id,confirmation=confirmation,
    expected_universe_sha256=universe_sha,expected_calendar_sha256=calendar_sha,
    assumed_latency_ms=1500,producer_code_commit=commit)
print('MICRO PREP PASS; observed bucket counts:',result.counts)
print('ZIP SHA-256:',result.sha256)
print('READ START event SHA-256:',result.event_sha256)


In [ ]:
# 4. Privatno preuzimanje. Bez Drive spremanja ZIP-a ili javnog GitHuba.
from google.colab import files
files.download(str(result.path))
print('Nakon preuzimanja: Runtime > Disconnect and delete runtime.')
